In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier

In [5]:
FEATURES_FILE = "features.csv"
TARGETS_FILE = "targets.csv"

ID_COL = "vk_id"
TARGET_COLS = ["Extraversion", "Agreeableness", "Conscientiousness", "Neuroticism", "Openness"]

In [6]:
# Загрузка и объединение
X_df = pd.read_csv(FEATURES_FILE)
y_df = pd.read_csv(TARGETS_FILE)

# Оставляем только id и target-колонки
y_df = y_df[[ID_COL] + TARGET_COLS]

# Склеиваем по id
df = X_df.merge(y_df, on=ID_COL, how="inner")
X = df.drop(columns=[ID_COL] + TARGET_COLS)
y = df[TARGET_COLS].copy()

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
def evaluate_multioutput_multiclass(y_true_df, y_pred_np, target_cols, model_name="Model"):
    y_true = y_true_df.to_numpy()
    y_pred = np.asarray(y_pred_np)

    exact_match_accuracy = np.mean(np.all(y_true == y_pred, axis=1))
    label_wise_accuracy = np.mean(y_true == y_pred)

    per_target_accuracy = []
    for i in range(len(target_cols)):
        acc_i = accuracy_score(y_true[:, i], y_pred[:, i])
        per_target_accuracy.append(acc_i)
    mean_target_accuracy = np.mean(per_target_accuracy)

    per_target_macro_f1 = []
    for i in range(len(target_cols)):
        f1_i = f1_score(y_true[:, i], y_pred[:, i], average="macro", zero_division=0)
        per_target_macro_f1.append(f1_i)
    mean_macro_f1 = np.mean(per_target_macro_f1)

    print(f"\n=== {model_name}: общий результат ===")
    print(f"Exact-match accuracy: {exact_match_accuracy:.4f}")
    print(f"Label-wise accuracy:  {label_wise_accuracy:.4f}")
    print(f"Mean target accuracy: {mean_target_accuracy:.4f}")
    print(f"Mean macro F1:        {mean_macro_f1:.4f}")

    print("\n=== Accuracy по target-колонкам ===")
    for col, score in zip(target_cols, per_target_accuracy):
        print(f"{col}: {score:.4f}")

    print("\n=== Macro F1 по target-колонкам ===")
    for col, score in zip(target_cols, per_target_macro_f1):
        print(f"{col}: {score:.4f}")

In [9]:
# Модель
rf_model = Pipeline(steps=[
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
evaluate_multioutput_multiclass(y_test, rf_pred, TARGET_COLS, model_name="Random Forest")


=== Random Forest: общий результат ===
Exact-match accuracy: 0.0169
Label-wise accuracy:  0.0746
Mean target accuracy: 0.0746
Mean macro F1:        0.0168

=== Accuracy по target-колонкам ===
Extraversion: 0.0169
Agreeableness: 0.1017
Conscientiousness: 0.0678
Neuroticism: 0.1017
Openness: 0.0847

=== Macro F1 по target-колонкам ===
Extraversion: 0.0034
Agreeableness: 0.0179
Conscientiousness: 0.0135
Neuroticism: 0.0264
Openness: 0.0230


In [10]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier

gb_model = Pipeline(steps=[
    ("model", MultiOutputClassifier(
        GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
    ))
])

gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

evaluate_multioutput_multiclass(y_test, gb_pred, TARGET_COLS, model_name="Gradient Boosting")


=== Gradient Boosting: общий результат ===
Exact-match accuracy: 0.0169
Label-wise accuracy:  0.0644
Mean target accuracy: 0.0644
Mean macro F1:        0.0213

=== Accuracy по target-колонкам ===
Extraversion: 0.1017
Agreeableness: 0.1017
Conscientiousness: 0.0508
Neuroticism: 0.0339
Openness: 0.0339

=== Macro F1 по target-колонкам ===
Extraversion: 0.0401
Agreeableness: 0.0429
Conscientiousness: 0.0113
Neuroticism: 0.0040
Openness: 0.0081


In [13]:
# CatBoost
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

cb_models = {}
cb_pred = np.empty((len(X_test_cb), len(TARGET_COLS)), dtype=object)

for i, target_col in enumerate(TARGET_COLS):
    n_classes = y_train[target_col].nunique()

    # если 2 класса -> Logloss, иначе MultiClass
    loss_function = "Logloss" if n_classes == 2 else "MultiClass"

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function=loss_function,
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_cb,
        y_train[target_col],
    )

    preds = model.predict(X_test_cb)
    cb_pred[:, i] = np.asarray(preds).ravel()
    cb_models[target_col] = model

# приведём типы предсказаний к типу y_test, если нужно
cb_pred = pd.DataFrame(cb_pred, columns=TARGET_COLS, index=y_test.index)

for col in TARGET_COLS:
    try:
        cb_pred[col] = cb_pred[col].astype(y_test[col].dtype)
    except Exception:
        pass

evaluate_multioutput_multiclass(y_test, cb_pred.to_numpy(), TARGET_COLS, model_name="CatBoost")


=== CatBoost: общий результат ===
Exact-match accuracy: 0.0169
Label-wise accuracy:  0.0644
Mean target accuracy: 0.0644
Mean macro F1:        0.0196

=== Accuracy по target-колонкам ===
Extraversion: 0.0339
Agreeableness: 0.0847
Conscientiousness: 0.0339
Neuroticism: 0.1186
Openness: 0.0508

=== Macro F1 по target-колонкам ===
Extraversion: 0.0096
Agreeableness: 0.0188
Conscientiousness: 0.0108
Neuroticism: 0.0352
Openness: 0.0238


# С графовым признаковым представлением

In [18]:
FEATURES_FILE = "features_with_graph.csv"
TARGETS_FILE = "targets.csv"

ID_COL = "vk_id"
TARGET_COLS = ["Extraversion", "Agreeableness", "Conscientiousness", "Neuroticism", "Openness"]

In [19]:
X_df = pd.read_csv(FEATURES_FILE)
y_df = pd.read_csv(TARGETS_FILE)

y_df = y_df[[ID_COL] + TARGET_COLS]

# если таргеты случайно есть в features, убираем их
X_df = X_df.drop(columns=[c for c in TARGET_COLS if c in X_df.columns], errors="ignore")

df = X_df.merge(y_df, on=ID_COL, how="inner")

X = df.drop(columns=[ID_COL] + TARGET_COLS)
y = df[TARGET_COLS].copy()

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [21]:
def evaluate_multioutput_multiclass(y_true_df, y_pred_np, target_cols, model_name="Model"):
    y_true = y_true_df.to_numpy()
    y_pred = np.asarray(y_pred_np)

    exact_match_accuracy = np.mean(np.all(y_true == y_pred, axis=1))
    label_wise_accuracy = np.mean(y_true == y_pred)

    per_target_accuracy = []
    for i in range(len(target_cols)):
        acc_i = accuracy_score(y_true[:, i], y_pred[:, i])
        per_target_accuracy.append(acc_i)
    mean_target_accuracy = np.mean(per_target_accuracy)

    per_target_macro_f1 = []
    for i in range(len(target_cols)):
        f1_i = f1_score(y_true[:, i], y_pred[:, i], average="macro", zero_division=0)
        per_target_macro_f1.append(f1_i)
    mean_macro_f1 = np.mean(per_target_macro_f1)

    print(f"\n=== {model_name}: общий результат ===")
    print(f"Exact-match accuracy: {exact_match_accuracy:.4f}")
    print(f"Label-wise accuracy:  {label_wise_accuracy:.4f}")
    print(f"Mean target accuracy: {mean_target_accuracy:.4f}")
    print(f"Mean macro F1:        {mean_macro_f1:.4f}")

    print("\n=== Accuracy по target-колонкам ===")
    for col, score in zip(target_cols, per_target_accuracy):
        print(f"{col}: {score:.4f}")

    print("\n=== Macro F1 по target-колонкам ===")
    for col, score in zip(target_cols, per_target_macro_f1):
        print(f"{col}: {score:.4f}")

In [22]:
# Модель
rf_model = Pipeline(steps=[
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
evaluate_multioutput_multiclass(y_test, rf_pred, TARGET_COLS, model_name="Random Forest")
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier

gb_model = Pipeline(steps=[
    ("model", MultiOutputClassifier(
        GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
    ))
])

gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

evaluate_multioutput_multiclass(y_test, gb_pred, TARGET_COLS, model_name="Gradient Boosting")
# CatBoost
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

cb_models = {}
cb_pred = np.empty((len(X_test_cb), len(TARGET_COLS)), dtype=object)

for i, target_col in enumerate(TARGET_COLS):
    n_classes = y_train[target_col].nunique()

    # если 2 класса -> Logloss, иначе MultiClass
    loss_function = "Logloss" if n_classes == 2 else "MultiClass"

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function=loss_function,
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_cb,
        y_train[target_col],
    )

    preds = model.predict(X_test_cb)
    cb_pred[:, i] = np.asarray(preds).ravel()
    cb_models[target_col] = model

# приведём типы предсказаний к типу y_test, если нужно
cb_pred = pd.DataFrame(cb_pred, columns=TARGET_COLS, index=y_test.index)

for col in TARGET_COLS:
    try:
        cb_pred[col] = cb_pred[col].astype(y_test[col].dtype)
    except Exception:
        pass

evaluate_multioutput_multiclass(y_test, cb_pred.to_numpy(), TARGET_COLS, model_name="CatBoost")


=== Random Forest: общий результат ===
Exact-match accuracy: 0.0169
Label-wise accuracy:  0.0712
Mean target accuracy: 0.0712
Mean macro F1:        0.0120

=== Accuracy по target-колонкам ===
Extraversion: 0.0508
Agreeableness: 0.1017
Conscientiousness: 0.0847
Neuroticism: 0.0847
Openness: 0.0339

=== Macro F1 по target-колонкам ===
Extraversion: 0.0125
Agreeableness: 0.0194
Conscientiousness: 0.0130
Neuroticism: 0.0061
Openness: 0.0090

=== Gradient Boosting: общий результат ===
Exact-match accuracy: 0.0000
Label-wise accuracy:  0.0475
Mean target accuracy: 0.0475
Mean macro F1:        0.0119

=== Accuracy по target-колонкам ===
Extraversion: 0.0508
Agreeableness: 0.0508
Conscientiousness: 0.0339
Neuroticism: 0.0678
Openness: 0.0339

=== Macro F1 по target-колонкам ===
Extraversion: 0.0202
Agreeableness: 0.0177
Conscientiousness: 0.0052
Neuroticism: 0.0077
Openness: 0.0086

=== CatBoost: общий результат ===
Exact-match accuracy: 0.0339
Label-wise accuracy:  0.0746
Mean target accurac